# Medication Prediction Analysis

This notebook implements machine learning models for predicting medication outcomes using:
- CatBoost gradient boosting
- Keras neural networks 
- Super Learner ensemble combining both models

The analysis includes internal validation on research cohorts and external validation across different datasets.

## Setup and Imports

In [1]:
# Standard library imports
import os
import pickle
import pandas as pd
import numpy as np
from typing import List, Dict
from sklearn.metrics import roc_auc_score

# Custom data loading functions
from utils.load_data import (
    load_ppp, load_pond, load_hbn, load_abcd, root_dir
)

# Custom model training and evaluation modules
from models.catboost_trainer import CatBoostTrainer
from models.keras_trainer import KerasTrainer
from models.logistic_trainer import LogisticTrainer
from models.super_learner import SuperLearner

# Utility modules
from utils.experiment_manager import (
    generate_experiment_data, 
    save_command_scripts
)
from utils.evaluation import (
    calculate_summary_statistics,
    format_confidence_interval,
    collect_super_learner_predictions,
    collect_model_predictions,
    average_fold_predictions,
    calculate_fairness_metrics,
    calculate_calibration,
    plot_roc_auc,
    plot_calibration,
    calculate_super_learner_shap_values, plot_shap_beeswarm
)

C:\Users\marle\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration Parameters

In [2]:
# Experimental design parameters
N_SUBSAMPLES = 10  # Number of balanced subsamples per outcome
N_FOLDS = 5        # Number of cross-validation folds

# Universal control variables
RUN_HYPERPARAMETER_TUNING = True  # Set to False to skip tuning and use defaults

# Random seed for reproducibility
RANDOM_SEED = 42

# Output directory sufficx
OUTPUT_SUFFIX = '_' + str(N_SUBSAMPLES) + 'bootstraps'
pond_output_dir = os.path.join(root_dir() + OUTPUT_SUFFIX, 'POND')
hbn_output_dir = os.path.join(root_dir() + OUTPUT_SUFFIX, 'HBN')
abcd_output_dir = os.path.join(root_dir() + OUTPUT_SUFFIX, 'ABCD')
ppp_output_dir = os.path.join(root_dir() + OUTPUT_SUFFIX, 'PPP')

## Research Cohort Analysis

### POND Dataset

#### Load Data

In [3]:
# Load POND dataset
pond_df, pond_features, pond_outcomes = load_pond()

# Identify continuous features (those with > 5 unique values)
pond_continuous_features = [
    feature for feature in pond_features 
    if pond_df[feature].nunique() > 5
]

print(f"POND Dataset:")
print(f"- {len(pond_df)} total samples")
print(f"- {len(pond_features)} features ({len(pond_continuous_features)} continuous)")
print(f"- {len(pond_outcomes)} outcome variables: {pond_outcomes}")

POND Dataset:
- 598 total samples
- 136 features (13 continuous)
- 4 outcome variables: ['stimulant_outcome', 'antidepressant_outcome', 'antipsychotic_outcome', 'nonstimulant_outcome']


#### Generate Hyperparameter Tuning Commands

In [4]:
# Generate balanced subsamples, CV splits, and tuning commands
tuning_commands = generate_experiment_data(
    df=pond_df,
    subject_col='subject',
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    base_dir=pond_output_dir
)

# Only generate and save tuning scripts if tuning is enabled
if RUN_HYPERPARAMETER_TUNING:
    save_command_scripts(tuning_commands, pond_output_dir)
    print("Run hyperparameter tuning using the generated shell scripts.")
    print("Set RUN_HYPERPARAMETER_TUNING = False to skip tuning and use default parameters.")
else:
    print("Hyperparameter tuning is disabled. Using default parameters:")
    print(f"- CatBoost: 100 iterations, 0.1 learning rate, depth 4")
    print(f"- Keras: 2 layers, 64 units, 0.001 learning rate")

Run: parallel --progress :::: A:/fedml-pilot-data_kushki/Marlee Vandewouw/revision/revision_10bootstraps\POND\run_catboost_tuning.sh


Run: parallel --progress :::: A:/fedml-pilot-data_kushki/Marlee Vandewouw/revision/revision_10bootstraps\POND\run_keras_tuning.sh
Run hyperparameter tuning using the generated shell scripts.
Set RUN_HYPERPARAMETER_TUNING = False to skip tuning and use default parameters.


#### Model Training and Internal Validation: Base Models

In [5]:
# Initialize model trainers
catboost_trainer = CatBoostTrainer(pond_output_dir)
keras_trainer = KerasTrainer(pond_output_dir)

print("Training and evaluating models on POND dataset...")

# Train base learners (needed for Super Learner)
pond_catboost_results, pond_catboost_tprs = catboost_trainer.train_and_evaluate_internal(
    df=pond_df,
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    use_tuned_params=RUN_HYPERPARAMETER_TUNING
)

pond_keras_results, pond_keras_tprs = keras_trainer.train_and_evaluate_internal(
    df=pond_df,
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    use_tuned_params=RUN_HYPERPARAMETER_TUNING
)

print("POND internal validation completed for base learners.")

Training and evaluating models on POND dataset...
Input validation passed: 598 samples, 136 features, 4 outcomes


Input validation passed: 598 samples, 136 features, 4 outcomes


POND internal validation completed for base learners.


#### Model Training and Internal Validation: Meta Learner

In [6]:
# Train Super Learner
super_learner = SuperLearner(pond_output_dir)
pond_super_learner_results, pond_super_learner_tprs = super_learner.train_and_evaluate_internal(
    df=pond_df,
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS
)

print("POND internal validation completed for super learners.")

Input validation passed: 598 samples, 136 features, 4 outcomes


POND internal validation completed for super learners.


#### Results - Super Learner

In [7]:
# Calculate summary statistics
metrics_to_analyze = ['ROC AUC', 'PR AUC', 'Accuracy', 'Sensitivity', 'Specificity']

print("=== POND Internal Validation Results - Super Learner ===")

summary = pond_super_learner_results.groupby('Outcome')[metrics_to_analyze].apply(
    lambda x: calculate_summary_statistics(x, metrics_to_analyze)
)

for outcome in pond_outcomes:
    if outcome in summary.index:
        row = summary.loc[outcome]
        formatted_auc = format_confidence_interval(
            row['ROC AUC_median'],
            row['ROC AUC_q1'],
            row['ROC AUC_q3']
        )
        print(f"\n{outcome}:")
        print(f"  ROC AUC: {formatted_auc}")

=== POND Internal Validation Results - Super Learner ===

stimulant_outcome:
  ROC AUC: 0.754 (0.729-0.798)

antidepressant_outcome:
  ROC AUC: 0.828 (0.782-0.869)

antipsychotic_outcome:
  ROC AUC: 0.790 (0.720-0.856)

nonstimulant_outcome:
  ROC AUC: 0.684 (0.638-0.779)


### External Validation - HBN Dataset

#### Load Data

In [8]:
# Load HBN dataset for external validation
hbn_df = load_hbn()

print(f"HBN Dataset: {len(hbn_df)} samples for external validation")

# Evaluate POND-trained models on HBN dataset
print("Evaluating POND models on HBN dataset...")

# Evaluate Super Learner
hbn_super_learner_results, hbn_super_learner_tprs = super_learner.evaluate_external(
    external_df=hbn_df,
    subject_col='EID',
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    trained_model_dir=pond_output_dir,
    output_dir=hbn_output_dir
)

print("HBN external validation completed.")

HBN Dataset: 1764 samples for external validation
Evaluating POND models on HBN dataset...
Input validation passed: 1764 samples, 136 features, 4 outcomes


HBN external validation completed.


#### Results - Super Learner

In [9]:
print("=== HBN External Validation Results - Super Learner ===")

summary = hbn_super_learner_results.groupby('Outcome')[metrics_to_analyze].apply(
    lambda x: calculate_summary_statistics(x, metrics_to_analyze)
)

for outcome in pond_outcomes:
    if outcome in summary.index:
        row = summary.loc[outcome]
        formatted_auc = format_confidence_interval(
            row['ROC AUC_median'],
            row['ROC AUC_q1'],
            row['ROC AUC_q3']
        )
        print(f"\n{outcome}:")
        print(f"  ROC AUC: {formatted_auc}")

=== HBN External Validation Results - Super Learner ===

stimulant_outcome:
  ROC AUC: 0.696 (0.674-0.711)

antidepressant_outcome:
  ROC AUC: 0.753 (0.737-0.774)

antipsychotic_outcome:
  ROC AUC: 0.794 (0.754-0.812)

nonstimulant_outcome:
  ROC AUC: 0.628 (0.592-0.654)


### External Validation - ABCD Dataset

#### Load Data

In [10]:
# Load ABCD dataset for external validation
abcd_df = load_abcd()

print(f"ABCD Dataset: {len(abcd_df)} samples for external validation")

# Evaluate POND-trained models on ABCD dataset
print("Evaluating POND models on ABCD dataset...")

# Evaluate Super Learner
abcd_super_learner_results, abcd_super_learner_tprs = super_learner.evaluate_external(
    external_df=abcd_df,
    subject_col='src_subject_id',
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    trained_model_dir=pond_output_dir,
    output_dir=abcd_output_dir
)

print("ABCD external validation completed.")

ABCD Dataset: 2396 samples for external validation
Evaluating POND models on ABCD dataset...
Input validation passed: 2396 samples, 136 features, 4 outcomes


ABCD external validation completed.


#### Results - Super Learner

In [11]:
print("=== ABCD External Validation Results - Super Learner ===")

summary = abcd_super_learner_results.groupby('Outcome')[metrics_to_analyze].apply(
    lambda x: calculate_summary_statistics(x, metrics_to_analyze)
)

for outcome in pond_outcomes:
    if outcome in summary.index:
        row = summary.loc[outcome]
        formatted_auc = format_confidence_interval(
            row['ROC AUC_median'],
            row['ROC AUC_q1'],
            row['ROC AUC_q3']
        )
        print(f"\n{outcome}:")
        print(f"  ROC AUC: {formatted_auc}")

=== ABCD External Validation Results - Super Learner ===

stimulant_outcome:
  ROC AUC: 0.702 (0.696-0.713)

antidepressant_outcome:
  ROC AUC: 0.673 (0.650-0.694)

antipsychotic_outcome:
  ROC AUC: 0.809 (0.787-0.832)

nonstimulant_outcome:
  ROC AUC: 0.655 (0.622-0.678)


#### Fairness

In [12]:
print("Collecting Super Learner predictions for the research cohorts...")

# Binary sensitive attributes
sensitive_cols = ['sex', 'intellectual_disability', 'White_Minority', 'income']
outcomes_to_analyze = ['stimulant_outcome', 'antidepressant_outcome', 'antipsychotic_outcome']

# === Cross-cohort analysis (POND + HBN + ABCD) ===
# 'indices_dir' selects each cohort's own balanced subsamples, while 'model_dir'
# stays with POND because that is where the models were trained. 'external' applies
# each fold's model to the entire balanced external sample, matching
# SuperLearner.evaluate_external, which writes no fold-level index files.
cross_cohort_data = [
    {
        'name': 'POND',
        'df': pond_df,
        'feature_cols': pond_features,
        'continuous_features': pond_continuous_features,
        'model_dir': pond_output_dir,
        'n_subsamples': N_SUBSAMPLES,
        'n_folds': N_FOLDS
    },
    {
        'name': 'HBN',
        'df': hbn_df,
        'feature_cols': pond_features,  # Using POND features for consistency
        'continuous_features': pond_continuous_features,
        'model_dir': pond_output_dir,  # Using POND-trained models
        'indices_dir': hbn_output_dir,  # Using HBN's own balanced subsamples
        'external': True,
        'n_subsamples': N_SUBSAMPLES,
        'n_folds': N_FOLDS
    },
    {
        'name': 'ABCD',
        'df': abcd_df,
        'feature_cols': pond_features,  # Using POND features for consistency
        'continuous_features': pond_continuous_features,
        'model_dir': pond_output_dir,  # Using POND-trained models
        'indices_dir': abcd_output_dir,  # Using ABCD's own balanced subsamples
        'external': True,
        'n_subsamples': N_SUBSAMPLES,
        'n_folds': N_FOLDS
    }
]

research_predictions = collect_super_learner_predictions(
    cohort_data_list=cross_cohort_data,
    outcome_vars=outcomes_to_analyze,
    extra_cols=sensitive_cols,
    cache_path=os.path.join(root_dir() + OUTPUT_SUFFIX, 'predictions_research.pkl')
)

# Average the fold predictions so each participant contributes once per subsample,
# rather than once per fold, which would weight the external cohorts by n_folds
research_predictions_avg = average_fold_predictions(
    research_predictions, extra_cols=sensitive_cols
)

cross_cohort_fairness = calculate_fairness_metrics(
    predictions=research_predictions_avg,
    sensitive_cols=sensitive_cols,
    outcome_vars=outcomes_to_analyze
)

print("Cross-Cohort Fairness Results:")
cross_cohort_fairness['Sensitive_Attribute'] = pd.Categorical(
    cross_cohort_fairness['Sensitive_Attribute'], categories=sensitive_cols, ordered=True
)
cross_cohort_fairness['Outcome'] = pd.Categorical(
    cross_cohort_fairness['Outcome'], categories=outcomes_to_analyze, ordered=True
)
print(cross_cohort_fairness.sort_values(['Sensitive_Attribute', 'Outcome']).round(3).to_string(index=False))

Loading cached predictions: A:/fedml-pilot-data_kushki/Marlee Vandewouw/revision/revision_10bootstraps\predictions_research.pkl


Cross-Cohort Fairness Results:
               Outcome     Sensitive_Attribute   DPR   EOR  N_total Group_0_label  N_group_0  Pct_pred_pos_group_0  TPR_group_0  FPR_group_0  Accuracy_group_0 Group_1_label  N_group_1  Pct_pred_pos_group_1  TPR_group_1  FPR_group_1  Accuracy_group_1 Group_2_label  N_group_2  Pct_pred_pos_group_2  TPR_group_2  FPR_group_2  Accuracy_group_2  DPR_pair_0_1  EOR_pair_0_1  DPR_pair_0_2  EOR_pair_0_2  DPR_pair_1_2  EOR_pair_1_2
     stimulant_outcome                     sex 0.745 0.686    17980           0.0      12529                61.849       74.461       46.587            64.937           1.0       5451                46.065       68.075       31.948            68.061           NaN          0                   NaN          NaN          NaN               NaN           NaN           NaN           NaN           NaN           NaN           NaN
antidepressant_outcome                     sex 0.698 0.614     6200           0.0       4137                33.406     

#### Fairness: Baseline

In [13]:
print("Calculating baseline fairness metrics from actual prescribing labels...")

# Uses the observed labels in place of the predictions, so the same participants
# and weighting are used as for the model-predicted ratios above
cross_cohort_baseline_fairness = calculate_fairness_metrics(
    predictions=research_predictions_avg,
    sensitive_cols=sensitive_cols,
    outcome_vars=outcomes_to_analyze,
    baseline=True
)

print("Cross-Cohort Baseline Fairness Results:")
cross_cohort_baseline_fairness['Sensitive_Attribute'] = pd.Categorical(
    cross_cohort_baseline_fairness['Sensitive_Attribute'], categories=sensitive_cols, ordered=True
)
cross_cohort_baseline_fairness['Outcome'] = pd.Categorical(
    cross_cohort_baseline_fairness['Outcome'], categories=outcomes_to_analyze, ordered=True
)
print(cross_cohort_baseline_fairness.sort_values(['Sensitive_Attribute', 'Outcome']).round(3).to_string(index=False))

Calculating baseline fairness metrics from actual prescribing labels...


Cross-Cohort Baseline Fairness Results:
               Outcome     Sensitive_Attribute   DPR  EOR  N_total Group_0_label  N_group_0  Pct_pred_pos_group_0  TPR_group_0  FPR_group_0  Accuracy_group_0 Group_1_label  N_group_1  Pct_pred_pos_group_1  TPR_group_1  FPR_group_1  Accuracy_group_1 Group_2_label  N_group_2  Pct_pred_pos_group_2  TPR_group_2  FPR_group_2  Accuracy_group_2  DPR_pair_0_1  EOR_pair_0_1  DPR_pair_0_2  EOR_pair_0_2  DPR_pair_1_2  EOR_pair_1_2
     stimulant_outcome                     sex 0.714  NaN    17980           0.0      12529                54.753        100.0          0.0             100.0           1.0       5451                39.075        100.0          0.0             100.0           NaN          0                   NaN          NaN          NaN               NaN           NaN           NaN           NaN           NaN           NaN           NaN
antidepressant_outcome                     sex 0.933  NaN     6200           0.0       4137                48.82

#### AU-ROC Figure

In [14]:
outcomes_to_plot = ['stimulant_outcome', 'antidepressant_outcome', 'antipsychotic_outcome']

# POND
plot_roc_auc(pond_output_dir, pond_super_learner_tprs[pond_super_learner_tprs['Outcome'].isin(outcomes_to_plot)])

# HBN
plot_roc_auc(hbn_output_dir, hbn_super_learner_tprs[hbn_super_learner_tprs['Outcome'].isin(outcomes_to_plot)])

# ABCD
plot_roc_auc(abcd_output_dir, abcd_super_learner_tprs[abcd_super_learner_tprs['Outcome'].isin(outcomes_to_plot)])


## Electronic Medical Records Analysis - PPP Dataset

#### Load Data

In [15]:
# Load PPP (EMR) dataset
ppp_df, ppp_features, ppp_outcomes, ppp_df_without_dummies = load_ppp()

# For EMR data, typically only age is continuous
ppp_continuous_features = ['age']

print(f"PPP EMR Dataset:")
print(f"- {len(ppp_df)} total samples")
print(f"- {len(ppp_features)} features ({len(ppp_continuous_features)} continuous)")
print(f"- {len(ppp_outcomes)} outcome variables: {ppp_outcomes}")

# Proportion of unknown
prop_unknown = (ppp_df_without_dummies == 'unknown').sum()/len(ppp_df_without_dummies)
prop_rows_unknown = (ppp_df_without_dummies == 'unknown').any(axis=1).mean()
print(f" - Max % unknown in a feature: {np.max(prop_unknown*100)}")
print(f" - % participants with at least one unknown: {prop_rows_unknown*100}")

PPP EMR Dataset:
- 312 total samples
- 104 features (1 continuous)
- 4 outcome variables: ['stimulant_outcome', 'antidepressant_outcome', 'antipsychotic_outcome', 'nonstimulant_outcome']
 - Max % unknown in a feature: 28.846153846153843
 - % participants with at least one unknown: 64.74358974358975


### Generate Hyperparameter Tuning Commands

In [16]:
# Generate balanced subsamples, CV splits, and tuning commands for PPP
ppp_tuning_commands = generate_experiment_data(
    df=ppp_df,
    subject_col='study_id2',
    feature_cols=ppp_features,
    continuous_features=ppp_continuous_features,
    outcome_vars=ppp_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    base_dir=ppp_output_dir
)

# Only generate and save tuning scripts if tuning is enabled
if RUN_HYPERPARAMETER_TUNING:
    save_command_scripts(ppp_tuning_commands, ppp_output_dir)
    print("Run hyperparameter tuning using the generated shell scripts.")
    print("Set RUN_HYPERPARAMETER_TUNING = False to skip tuning and use default parameters.")
else:
    print("Hyperparameter tuning is disabled. Using default parameters:")
    print(f"- CatBoost: 100 iterations, 0.1 learning rate, depth 4")
    print(f"- Keras: 2 layers, 64 units, 0.001 learning rate")

Run: parallel --progress :::: A:/fedml-pilot-data_kushki/Marlee Vandewouw/revision/revision_10bootstraps\PPP\run_catboost_tuning.sh


Run: parallel --progress :::: A:/fedml-pilot-data_kushki/Marlee Vandewouw/revision/revision_10bootstraps\PPP\run_keras_tuning.sh
Run hyperparameter tuning using the generated shell scripts.
Set RUN_HYPERPARAMETER_TUNING = False to skip tuning and use default parameters.


### Model Training and Internal Validation: Base Learners

In [17]:
# Initialize model trainers for PPP dataset
ppp_catboost_trainer = CatBoostTrainer(ppp_output_dir)
ppp_keras_trainer = KerasTrainer(ppp_output_dir)

print("Training and evaluating models on PPP EMR dataset...")

# Train base learners (needed for Super Learner)
ppp_catboost_results, ppp_catboost_tprs = ppp_catboost_trainer.train_and_evaluate_internal(
    df=ppp_df,
    feature_cols=ppp_features,
    continuous_features=ppp_continuous_features,
    outcome_vars=ppp_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    use_tuned_params=RUN_HYPERPARAMETER_TUNING
)

ppp_keras_results, ppp_keras_tprs = ppp_keras_trainer.train_and_evaluate_internal(
    df=ppp_df,
    feature_cols=ppp_features,
    continuous_features=ppp_continuous_features,
    outcome_vars=ppp_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    use_tuned_params=RUN_HYPERPARAMETER_TUNING
)

print("PPP EMR internal validation completed for base learners.")

Training and evaluating models on PPP EMR dataset...
Input validation passed: 312 samples, 104 features, 4 outcomes


Input validation passed: 312 samples, 104 features, 4 outcomes


PPP EMR internal validation completed for base learners.


### Model Training and Internal Validation: Meta Learner

In [18]:
# Train Super Learner
ppp_super_learner = SuperLearner(ppp_output_dir)
ppp_super_learner_results, ppp_super_learner_tprs = ppp_super_learner.train_and_evaluate_internal(
    df=ppp_df,
    feature_cols=ppp_features,
    continuous_features=ppp_continuous_features,
    outcome_vars=ppp_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS
)

print("PPP EMR internal validation completed for meta learner.")

Input validation passed: 312 samples, 104 features, 4 outcomes


PPP EMR internal validation completed for meta learner.


### Results - Super Learner

In [19]:
print("=== PPP EMR Internal Validation Results - Super Learner ===")

summary = ppp_super_learner_results.groupby('Outcome')[metrics_to_analyze].apply(
    lambda x: calculate_summary_statistics(x, metrics_to_analyze)
)

for outcome in ppp_outcomes:
    if outcome in summary.index:
        row = summary.loc[outcome]
        formatted_auc = format_confidence_interval(
            row['ROC AUC_median'],
            row['ROC AUC_q1'],
            row['ROC AUC_q3']
        )
        print(f"\n{outcome}:")
        print(f"  ROC AUC: {formatted_auc}")

=== PPP EMR Internal Validation Results - Super Learner ===

stimulant_outcome:
  ROC AUC: 0.842 (0.809-0.881)

antidepressant_outcome:
  ROC AUC: 0.824 (0.771-0.874)

antipsychotic_outcome:
  ROC AUC: 0.873 (0.829-0.905)

nonstimulant_outcome:
  ROC AUC: 0.803 (0.752-0.831)


### Fairness

In [20]:
print("Collecting Super Learner predictions for the EMR cohort...")

# Binary sensitive attributes
sensitive_cols_ppp = ['sex', 'intellectual_disability']

# PPP is internally validated, so each fold model is evaluated on its own held-out
# validation rows and no averaging across folds is required
ppp_data = [
    {
        'name': 'PPP',
        'df': ppp_df,
        'feature_cols': ppp_features,
        'continuous_features': ppp_continuous_features,
        'model_dir': ppp_output_dir,
        'n_subsamples': N_SUBSAMPLES,
        'n_folds': N_FOLDS
    }
]

ppp_predictions = collect_super_learner_predictions(
    cohort_data_list=ppp_data,
    outcome_vars=outcomes_to_analyze,
    extra_cols=sensitive_cols_ppp,
    cache_path=os.path.join(root_dir() + OUTPUT_SUFFIX, 'predictions_emr.pkl')
)

ppp_fairness = calculate_fairness_metrics(
    predictions=ppp_predictions,
    sensitive_cols=sensitive_cols_ppp,
    outcome_vars=outcomes_to_analyze
)

print("PPP Fairness Results:")
ppp_fairness['Sensitive_Attribute'] = pd.Categorical(
    ppp_fairness['Sensitive_Attribute'], categories=sensitive_cols_ppp, ordered=True
)
ppp_fairness['Outcome'] = pd.Categorical(
    ppp_fairness['Outcome'], categories=outcomes_to_analyze, ordered=True
)
print(ppp_fairness.sort_values(['Sensitive_Attribute', 'Outcome']).round(3).to_string(index=False))

Loading cached predictions: A:/fedml-pilot-data_kushki/Marlee Vandewouw/revision/revision_10bootstraps\predictions_emr.pkl


PPP Fairness Results:
               Outcome     Sensitive_Attribute   DPR   EOR  N_total Group_0_label  N_group_0  Pct_pred_pos_group_0  TPR_group_0  FPR_group_0  Accuracy_group_0 Group_1_label  N_group_1  Pct_pred_pos_group_1  TPR_group_1  FPR_group_1  Accuracy_group_1 Group_2_label  N_group_2  Pct_pred_pos_group_2  TPR_group_2  FPR_group_2  Accuracy_group_2  DPR_pair_0_1  EOR_pair_0_1  DPR_pair_0_2  EOR_pair_0_2  DPR_pair_1_2  EOR_pair_1_2
     stimulant_outcome                     sex 0.790 0.489     2280        female        345                36.232       76.429        8.780            85.217          male       1935                45.840       71.900       17.968            76.796           NaN          0                   NaN          NaN          NaN               NaN           NaN           NaN           NaN           NaN           NaN           NaN
antidepressant_outcome                     sex 0.791 0.723     1640        female        281                55.160       75.625 

### Fairness: Baseline

In [21]:
print("Calculating baseline fairness metrics from actual prescribing labels...")

ppp_baseline_fairness = calculate_fairness_metrics(
    predictions=ppp_predictions,
    sensitive_cols=sensitive_cols_ppp,
    outcome_vars=outcomes_to_analyze,
    baseline=True
)

ppp_baseline_fairness['Sensitive_Attribute'] = pd.Categorical(
    ppp_baseline_fairness['Sensitive_Attribute'], categories=sensitive_cols_ppp, ordered=True
)
ppp_baseline_fairness['Outcome'] = pd.Categorical(
    ppp_baseline_fairness['Outcome'], categories=outcomes_to_analyze, ordered=True
)
print("PPP Baseline Fairness Results:")
print(ppp_baseline_fairness.sort_values(['Sensitive_Attribute', 'Outcome']).round(3).to_string(index=False))

Calculating baseline fairness metrics from actual prescribing labels...


PPP Baseline Fairness Results:
               Outcome     Sensitive_Attribute   DPR  EOR  N_total Group_0_label  N_group_0  Pct_pred_pos_group_0  TPR_group_0  FPR_group_0  Accuracy_group_0 Group_1_label  N_group_1  Pct_pred_pos_group_1  TPR_group_1  FPR_group_1  Accuracy_group_1 Group_2_label  N_group_2  Pct_pred_pos_group_2  TPR_group_2  FPR_group_2  Accuracy_group_2  DPR_pair_0_1  EOR_pair_0_1  DPR_pair_0_2  EOR_pair_0_2  DPR_pair_1_2  EOR_pair_1_2
     stimulant_outcome                     sex 0.785  NaN     2280        female        345                40.580        100.0          0.0             100.0          male       1935                51.680        100.0          0.0             100.0           NaN          0                   NaN          NaN          NaN               NaN           NaN           NaN           NaN           NaN           NaN           NaN
antidepressant_outcome                     sex 0.853  NaN     1640        female        281                56.940        

### AU-ROC Figure

In [22]:
outcomes_to_plot = ['stimulant_outcome', 'antidepressant_outcome', 'antipsychotic_outcome']

# PPP
plot_roc_auc(ppp_output_dir, ppp_super_learner_tprs[ppp_super_learner_tprs['Outcome'].isin(outcomes_to_plot)])

### SHAP

In [23]:
# Process each outcome
for outcome in ['stimulant_outcome', 'antidepressant_outcome', 'antipsychotic_outcome']:
    print(f"\nProcessing {outcome}")
    
    try:
        # Calculate SHAP values using correct pipeline
        shap_df = calculate_super_learner_shap_values(
            df=ppp_df,
            feature_cols=ppp_features,
            continuous_features=ppp_continuous_features,
            outcome_var=outcome,
            model_dir=ppp_output_dir,
            n_subsamples=N_SUBSAMPLES,
            n_folds=N_FOLDS
        )
        
        print(f"SHAP DataFrame shape: {shap_df.shape}")
        print(f"Samples with SHAP values: {(shap_df['count'] > 0).sum()}")
        
        # Create beeswarm plot
        plot_shap_beeswarm(
            shap_df=shap_df,
            df=ppp_df,
            feature_cols=ppp_features,
            outcome_var=outcome,
            save_dir=ppp_output_dir,
            top_n=10
        )
        
        # Save SHAP DataFrame
        shap_df.to_csv(os.path.join(ppp_output_dir, f'{outcome}_shap_values.csv'), index=False)
        
        print(f"Completed {outcome}")
        
    except Exception as e:
        print(f"Error processing {outcome}: {e}")
        continue

print(f"\nSHAP analysis complete!")
print(f"Results saved to: {ppp_output_dir}")


Processing stimulant_outcome
Computing Super Learner SHAP values for stimulant_outcome
Processing subsample 1/10
  Processing fold 1/5


  Processing fold 2/5


  Processing fold 3/5


  Processing fold 4/5


  Processing fold 5/5


...

  Processing fold 2/5
  Processing fold 3/5


  Processing fold 4/5


  Processing fold 5/5
SHAP calculation complete. 312 samples have SHAP values.
SHAP DataFrame shape: (312, 105)
Samples with SHAP values: 312


SHAP beeswarm plot saved: A:/fedml-pilot-data_kushki/Marlee Vandewouw/revision/revision_10bootstraps\PPP\antipsychotic_outcome_shap.png


Completed antipsychotic_outcome

SHAP analysis complete!
Results saved to: A:/fedml-pilot-data_kushki/Marlee Vandewouw/revision/revision_10bootstraps\PPP


## Results Summary

In [24]:
print("\n" + "="*60)
print("COMPREHENSIVE RESULTS SUMMARY - SUPER LEARNER ONLY")
print("="*60)

datasets_and_results = [
    ("POND Research Cohort (Internal)", pond_super_learner_results, pond_outcomes),
    ("HBN Research Cohort (External)", hbn_super_learner_results, pond_outcomes),
    ("ABCD Research Cohort (External)", abcd_super_learner_results, pond_outcomes),
    ("PPP EMR Cohort (Internal)", ppp_super_learner_results, ppp_outcomes)
]

for dataset_name, results, outcomes in datasets_and_results:
    print(f"\n{dataset_name}:")
    
    if len(results) > 0:
        summary = results.groupby('Outcome')[metrics_to_analyze].apply(
            lambda x: calculate_summary_statistics(x, metrics_to_analyze)
        )
        
        for outcome in outcomes:
            if outcome in summary.index:
                row = summary.loc[outcome]
                formatted_auc = format_confidence_interval(
                    row['ROC AUC_median'],
                    row['ROC AUC_q1'],
                    row['ROC AUC_q3']
                )
                print(f"  {outcome}: {formatted_auc}")
    else:
        print("  No results available")


COMPREHENSIVE RESULTS SUMMARY - SUPER LEARNER ONLY

POND Research Cohort (Internal):
  stimulant_outcome: 0.754 (0.729-0.798)
  antidepressant_outcome: 0.828 (0.782-0.869)
  antipsychotic_outcome: 0.790 (0.720-0.856)
  nonstimulant_outcome: 0.684 (0.638-0.779)

HBN Research Cohort (External):
  stimulant_outcome: 0.696 (0.674-0.711)
  antidepressant_outcome: 0.753 (0.737-0.774)
  antipsychotic_outcome: 0.794 (0.754-0.812)
  nonstimulant_outcome: 0.628 (0.592-0.654)

ABCD Research Cohort (External):
  stimulant_outcome: 0.702 (0.696-0.713)
  antidepressant_outcome: 0.673 (0.650-0.694)
  antipsychotic_outcome: 0.809 (0.787-0.832)
  nonstimulant_outcome: 0.655 (0.622-0.678)

PPP EMR Cohort (Internal):
  stimulant_outcome: 0.842 (0.809-0.881)
  antidepressant_outcome: 0.824 (0.771-0.874)
  antipsychotic_outcome: 0.873 (0.829-0.905)
  nonstimulant_outcome: 0.803 (0.752-0.831)


## Secondary Results Summary

In [25]:
print("\n" + "="*60)
print("SECONDARY RESULTS SUMMARY - SUPER LEARNER ONLY")
print("="*60)

secondary_metrics = ['PR AUC', 'Sensitivity', 'Specificity']

datasets_and_results = [
    ("POND Research Cohort (Internal)", pond_super_learner_results, pond_outcomes),
    ("HBN Research Cohort (External)", hbn_super_learner_results, pond_outcomes),
    ("ABCD Research Cohort (External)", abcd_super_learner_results, pond_outcomes),
    ("PPP EMR Cohort (Internal)", ppp_super_learner_results, ppp_outcomes)
]

for dataset_name, results, outcomes in datasets_and_results:
    print(f"\n{dataset_name}:")

    if len(results) > 0:
        summary = results.groupby('Outcome')[metrics_to_analyze].apply(
            lambda x: calculate_summary_statistics(x, metrics_to_analyze)
        )

        for outcome in outcomes:
            if outcome in summary.index:
                row = summary.loc[outcome]
                print(f"  {outcome}:")
                for metric in secondary_metrics:
                    formatted = format_confidence_interval(
                        row[f'{metric}_median'],
                        row[f'{metric}_q1'],
                        row[f'{metric}_q3']
                    )
                    print(f"    {metric}: {formatted}")
    else:
        print("  No results available")


SECONDARY RESULTS SUMMARY - SUPER LEARNER ONLY

POND Research Cohort (Internal):
  stimulant_outcome:
    PR AUC: 0.762 (0.724-0.797)
    Sensitivity: 0.710 (0.650-0.770)
    Specificity: 0.676 (0.604-0.735)
  antidepressant_outcome:
    PR AUC: 0.825 (0.771-0.885)
    Sensitivity: 0.789 (0.684-0.842)
    Specificity: 0.737 (0.684-0.797)
  antipsychotic_outcome:
    PR AUC: 0.825 (0.747-0.877)
    Sensitivity: 0.714 (0.609-0.800)
    Specificity: 0.700 (0.500-0.800)
  nonstimulant_outcome:
    PR AUC: 0.719 (0.649-0.807)
    Sensitivity: 0.727 (0.568-0.818)
    Specificity: 0.564 (0.364-0.801)

HBN Research Cohort (External):
  stimulant_outcome:
    PR AUC: 0.664 (0.647-0.686)
    Sensitivity: 0.766 (0.719-0.814)
    Specificity: 0.503 (0.453-0.548)
  antidepressant_outcome:
    PR AUC: 0.748 (0.734-0.767)
    Sensitivity: 0.493 (0.472-0.542)
    Specificity: 0.840 (0.819-0.861)
  antipsychotic_outcome:
    PR AUC: 0.777 (0.735-0.810)
    Sensitivity: 0.692 (0.538-0.769)
    Specific

## Per-Group Fairness Rates (Supplemental Tables 8 and 9)

In [26]:
print("Building the per-group rate tables...")


def build_per_group_table(fairness_results):
    """Reshape the fairness output into the per-group layout reported in the supplement."""
    rows = []

    for _, result in fairness_results.iterrows():
        for i in range(3):
            if pd.isna(result[f'Group_{i}_label']):
                continue
            rows.append({
                'Attribute': result['Sensitive_Attribute'],
                'Outcome': result['Outcome'],
                'Group': result[f'Group_{i}_label'],
                'N': result[f'N_group_{i}'],
                'PP (%)': result[f'Pct_pred_pos_group_{i}'],
                'TPR (%)': result[f'TPR_group_{i}'],
                'FPR (%)': result[f'FPR_group_{i}'],
                'Accuracy (%)': result[f'Accuracy_group_{i}']
            })

    return pd.DataFrame(rows)


research_per_group = build_per_group_table(
    cross_cohort_fairness.sort_values(['Sensitive_Attribute', 'Outcome'])
)
ppp_per_group = build_per_group_table(
    ppp_fairness.sort_values(['Sensitive_Attribute', 'Outcome'])
)

print("\nResearch cohorts (Supplemental Table 8):")
print(research_per_group.round(1).to_string(index=False))

print("\nEMR cohort (Supplemental Table 9):")
print(ppp_per_group.round(1).to_string(index=False))

Building the per-group rate tables...

Research cohorts (Supplemental Table 8):
              Attribute                Outcome    Group     N  PP (%)  TPR (%)  FPR (%)  Accuracy (%)
                    sex      stimulant_outcome      0.0 12529    61.8     74.5     46.6          64.9
                    sex      stimulant_outcome      1.0  5451    46.1     68.1     31.9          68.1
                    sex antidepressant_outcome      0.0  4137    33.4     49.7     17.9          66.3
                    sex antidepressant_outcome      1.0  2063    47.9     65.0     29.1          67.8
                    sex  antipsychotic_outcome      0.0  1779    46.8     65.0     25.4          69.4
                    sex  antipsychotic_outcome      1.0   641    41.7     74.8     20.5          77.7
intellectual_disability      stimulant_outcome    False 16638    57.8     74.0     41.7          66.1
intellectual_disability      stimulant_outcome     True  1342    47.8     60.7     34.2          63.2
in

## Calibration (Supplemental Table 4, Supplemental Figure 1)

In [27]:
print("Calculating calibration...")

# Calibration is assessed on the fold-level predictions rather than the fold-averaged
# ones, so that every fold model contributes
all_predictions = pd.concat([research_predictions, ppp_predictions], ignore_index=True)

calibration_metrics, calibration_curves = calculate_calibration(all_predictions)

calibration_metrics['Cohort'] = pd.Categorical(
    calibration_metrics['Cohort'], categories=['POND', 'HBN', 'ABCD', 'PPP'], ordered=True
)
calibration_metrics['Outcome'] = pd.Categorical(
    calibration_metrics['Outcome'], categories=outcomes_to_analyze, ordered=True
)

print("Calibration Results (Supplemental Table 4):")
print(calibration_metrics.sort_values(['Cohort', 'Outcome']).round(3).to_string(index=False))

# Calibration curves (Supplemental Figure 1)
for cohort, output_dir in [('POND', pond_output_dir), ('HBN', hbn_output_dir),
                           ('ABCD', abcd_output_dir), ('PPP', ppp_output_dir)]:
    plot_calibration(output_dir, calibration_curves, cohort)

Calculating calibration...


Calibration Results (Supplemental Table 4):
Cohort                Outcome  Slope  Intercept  Brier     N
  POND      stimulant_outcome  1.230     -0.027  0.204  3420
  POND antidepressant_outcome  1.397     -0.056  0.178  1920
  POND  antipsychotic_outcome  1.701     -0.066  0.210  1040
   HBN      stimulant_outcome  0.978     -0.239  0.227 17700
   HBN antidepressant_outcome  1.054      0.415  0.213  7200
   HBN  antipsychotic_outcome  1.791      0.082  0.211  2600
  ABCD      stimulant_outcome  0.988     -0.051  0.221 55100
  ABCD antidepressant_outcome  0.712      0.400  0.240 14200
  ABCD  antipsychotic_outcome  2.032      0.155  0.203  4300
   PPP      stimulant_outcome  1.422     -0.058  0.163  2280
   PPP antidepressant_outcome  1.666     -0.059  0.183  1640
   PPP  antipsychotic_outcome  1.342     -0.061  0.143  2460


## Model Comparison (Supplemental Table 5)

In [28]:
print("Comparing the Super Learner with its base learners and logistic regression...")

# Base learner and ensemble predictions, collected in a single pass because loading
# the Keras models is the expensive step
model_predictions = collect_model_predictions(
    cohort_data_list=cross_cohort_data + ppp_data,
    outcome_vars=outcomes_to_analyze,
    cache_path=os.path.join(root_dir() + OUTPUT_SUFFIX, 'predictions_ablation.pkl')
)

# AU-ROC per cohort, outcome and fold for the ensemble and each base learner
model_auc = []
for (cohort, outcome, subsample, fold), group in model_predictions.groupby(
    ['Cohort', 'Outcome', 'Subsample', 'Fold'], observed=True
):
    for column, model_name in [('prob_super', 'Super Learner'),
                               ('prob_catboost', 'CatBoost'),
                               ('prob_keras', 'Neural network')]:
        model_auc.append({
            'Cohort': cohort,
            'Outcome': outcome,
            'Model': model_name,
            'ROC AUC': roc_auc_score(group['y_true'], group[column])
        })

# Logistic regression comparator, using the same subsamples, folds and scaling
pond_logistic_trainer = LogisticTrainer(pond_output_dir)
ppp_logistic_trainer = LogisticTrainer(ppp_output_dir)

logistic_results = []

pond_logistic, _ = pond_logistic_trainer.train_and_evaluate_internal(
    df=pond_df,
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=outcomes_to_analyze,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS
)
pond_logistic['Cohort'] = 'POND'
logistic_results.append(pond_logistic)

for cohort_name, external_df, external_dir in [('HBN', hbn_df, hbn_output_dir),
                                               ('ABCD', abcd_df, abcd_output_dir)]:
    external_logistic, _ = pond_logistic_trainer.evaluate_external(
        external_df=external_df,
        feature_cols=pond_features,
        continuous_features=pond_continuous_features,
        outcome_vars=outcomes_to_analyze,
        n_subsamples=N_SUBSAMPLES,
        n_folds=N_FOLDS,
        train_df=pond_df,
        trained_model_dir=pond_output_dir,
        output_dir=external_dir
    )
    external_logistic['Cohort'] = cohort_name
    logistic_results.append(external_logistic)

ppp_logistic, _ = ppp_logistic_trainer.train_and_evaluate_internal(
    df=ppp_df,
    feature_cols=ppp_features,
    continuous_features=ppp_continuous_features,
    outcome_vars=outcomes_to_analyze,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS
)
ppp_logistic['Cohort'] = 'PPP'
logistic_results.append(ppp_logistic)

logistic_auc = pd.concat(logistic_results, ignore_index=True)
logistic_auc['Model'] = 'Logistic regression'

model_comparison = pd.concat(
    [pd.DataFrame(model_auc), logistic_auc[['Cohort', 'Outcome', 'Model', 'ROC AUC']]],
    ignore_index=True
)

model_summary = model_comparison.groupby(['Cohort', 'Outcome', 'Model'])[['ROC AUC']].apply(
    lambda x: calculate_summary_statistics(x, ['ROC AUC'])
)

print("\nModel Comparison (Supplemental Table 5):")
models_to_compare = ['Super Learner', 'CatBoost', 'Neural network', 'Logistic regression']
print(f"  {'Cohort':<7}{'Outcome':<24}" + ''.join(m.rjust(22) for m in models_to_compare))

for cohort in ['POND', 'HBN', 'ABCD', 'PPP']:
    for outcome in outcomes_to_analyze:
        line = f"  {cohort:<7}{outcome:<24}"
        for model_name in models_to_compare:
            row = model_summary.loc[(cohort, outcome, model_name)]
            line += format_confidence_interval(
                row['ROC AUC_median'], row['ROC AUC_q1'], row['ROC AUC_q3'], decimals=2
            ).rjust(22)
        print(line)

Comparing the Super Learner with its base learners and logistic regression...
Loading cached model predictions: A:/fedml-pilot-data_kushki/Marlee Vandewouw/revision/revision_10bootstraps\predictions_ablation.pkl


Input validation passed: 598 samples, 136 features, 3 outcomes


Input validation passed: 1764 samples, 136 features, 3 outcomes


Input validation passed: 2396 samples, 136 features, 3 outcomes


Input validation passed: 312 samples, 104 features, 3 outcomes



Model Comparison (Supplemental Table 5):
  Cohort Outcome                          Super Learner              CatBoost        Neural network   Logistic regression
  POND   stimulant_outcome             0.75 (0.73-0.80)      0.75 (0.71-0.78)      0.75 (0.72-0.79)      0.67 (0.63-0.70)
  POND   antidepressant_outcome        0.83 (0.78-0.87)      0.81 (0.76-0.86)      0.81 (0.77-0.86)      0.75 (0.68-0.81)
  POND   antipsychotic_outcome         0.79 (0.72-0.86)      0.75 (0.70-0.81)      0.80 (0.72-0.85)      0.74 (0.68-0.81)
  HBN    stimulant_outcome             0.70 (0.67-0.71)      0.69 (0.68-0.71)      0.67 (0.64-0.69)      0.64 (0.61-0.66)
  HBN    antidepressant_outcome        0.75 (0.74-0.77)      0.75 (0.74-0.77)      0.73 (0.71-0.75)      0.71 (0.67-0.73)
  HBN    antipsychotic_outcome         0.79 (0.75-0.81)      0.78 (0.74-0.81)      0.76 (0.71-0.80)      0.76 (0.73-0.78)
  ABCD   stimulant_outcome             0.70 (0.70-0.71)      0.69 (0.68-0.70)      0.69 (0.67-0.70)     